<a href="https://colab.research.google.com/github/pj-pj-pj/legalbert-case-classifier/blob/feat%2Fbaseline-binary-classifier/legalbert_case_classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# install required libraries
!pip install transformers datasets torch scikit-learn pandas imbalanced-learn

ERROR: Operation cancelled by user
^C


In [ ]:
# import libs
import torch, pandas as pd, numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset
from sklearn.metrics import accuracy_score, f1_score

In [ ]:
df = pd.read_csv("cases.csv")

df.head(3) # civil = 0, criminal = 1

In [ ]:
# data cleaning/preprocessing
import re
from tqdm import tqdm  # progress bars
tqdm.pandas()

# func
def clean_legal_text(text):
    """
    Comprehensive legal text cleaning function
    """
    if not isinstance(text, str):
        return ""

    # 1. Handle line breaks and tabs
    text = text.replace('\n', ' ').replace('\r', ' ').replace('\t', ' ')

    # 2. Remove dates
    date_patterns = [
        r'\b\d{1,2}[-/]\d{1,2}[-/]\d{2,4}\b',  # DD-MM-YYYY
        r'\b\d{4}[-/]\d{1,2}[-/]\d{1,2}\b',    # YYYY-MM-DD
        r'\b(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)[a-z]* \d{1,2},? \d{4}\b',
    ]
    for pattern in date_patterns:
        text = re.sub(pattern, '[DATE]', text, flags=re.IGNORECASE)

    # 3. Preserve legal punctuation but remove excessive special chars
    # Keep: . , ; : ? ! ( ) - [ ] { } " ' § ¶ @ # $ % & * + = < >
    text = re.sub(r'[^\w\s.,;:?!()\-\[\]{}"\'§¶@#$%&*+=<>]', ' ', text)

    # 4. Normalize whitespace
    text = re.sub(r'\s+', ' ', text).strip()

    # removes numbers (keeps only words)
    text = ' '.join(re.findall(r"\d*[a-zA-Z][\w]*", text))

    return text



# call func
if 'text' in df.columns:
    df['cleaned_text'] = df['text'].apply(clean_legal_text)

    # Save result
    df.to_csv("cases_cleaned.csv", index=False)
    print("Cleaning complete! Saved to cases_cleaned.csv")

    # Show samples
    for i in range(2):
        print(f"\nSample {i}:")
        print(df['cleaned_text'].iloc[i][:300])

df.head(3)

In [ ]:
# stratified train-test split or stratified sampling (makes train and test split identical)
# convert to pands, stratified split, convert back

cleaned_df = pd.read_csv("cases_cleaned.csv")

from sklearn.model_selection import train_test_split
train_df, test_df = train_test_split(
    cleaned_df,
    test_size=0.25,
    random_state=42,
    stratify=df["label"]
)

print(f"\nTrain:")
print(train_df["label"].value_counts(normalize=True))
print(f"\nTest:")
print(test_df["label"].value_counts(normalize=True))

# Convert
# train_dataset = Dataset.from_pandas(train_df.reset_index(drop=True))
# eval_dataset = Dataset.from_pandas(test_df.reset_index(drop=True))

In [ ]:
#oversample bc data is imbalanced
# from imblearn.over_sampling import RandomOverSampler

# X_train = train_df['cleaned_text'].values.reshape(-1, 1)
# y_train = train_df['label'].values

# ros = RandomOverSampler(random_state=42)
# X_resampled, y_resampled = ros.fit_resample(X_train, y_train)

# # Create balanced DataFrame
# balanced_train_df = pd.DataFrame({
#     'cleaned_text': X_resampled.ravel(),  # Convert back to 1D
#     'label': y_resampled
# })

# print("\nTrain distribution (after oversampling):")
# print(balanced_train_df['label'].value_counts())

# print("\nTest distribution (unchanged - real data):")
# print(test_df['label'].value_counts())

In [ ]:
# encode labels

from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()
all_labels = pd.concat([train_df['label'], test_df['label']])
label_encoder.fit(all_labels)

train_df["label_encoded"] = label_encoder.transform(train_df["label"])
test_df["label_encoded"] = label_encoder.transform(test_df["label"])

label_encoder.classes_

In [ ]:
# convert to huggingface dataset

from datasets import Dataset

train_dataset = Dataset.from_pandas(train_df)
eval_dataset = Dataset.from_pandas(test_df)

In [ ]:
# load lb tokenizer

from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    "nlpaueb/legal-bert-base-uncased"
)

In [ ]:
# tokenize

def tokenize_function(examples):
    return tokenizer(
        examples["cleaned_text"],
        truncation=True,
        padding="max_length",
        max_length=512
    )


print("Available columns in train_dataset:")
print(train_dataset.column_names)

print("\nAvailable columns in eval_dataset:")
print(eval_dataset.column_names)

# Or check first row
print("\nFirst sample keys:")
print(train_dataset[0].keys() if isinstance(train_dataset[0], dict) else "Not a dict")

train_tokenized = train_dataset.map(tokenize_function, batched=True)
eval_tokenized = eval_dataset.map(tokenize_function, batched=True)


In [ ]:
# load legalbert

from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "nlpaueb/legal-bert-base-uncased",
    num_labels=2 # 2 for Bi-Classification, 3 otherwise (with legal fees)
)


In [ ]:
# training config

from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./legalbert_model",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=3, # change from 3-5
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=10,
    save_strategy="no"
)


In [ ]:
# metrics

from sklearn.metrics import accuracy_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, predictions)
    }


In [ ]:
from sklearn.utils.class_weight import compute_class_weight

weights = compute_class_weight(
    class_weight = 'balanced',
    classes=np.unique(train_df["label_encoded"]),
    y=train_df["label_encoded"]
)

# 2. Convert to a Tensor and send to GPU/CPU
device = "cuda" if torch.cuda.is_available() else "cpu"
class_weights = torch.tensor(weights, dtype=torch.float).to(device)

print(f"Weight for Civil (0): {weights[0]}")
print(f"Weight for Criminal (1): {weights[1]}")

In [ ]:
import torch.nn as nn

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        # Get predictions from Legal-BERT
        outputs = model(**inputs)
        logits = outputs.get("logits")

        # Tell the model to use the weights we calculated earlier
        # 'class_weights'
        loss_fct = nn.CrossEntropyLoss(weight=class_weights)

        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

In [ ]:
# train

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=eval_tokenized,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

trainer.train()


In [ ]:
from transformers import pipeline

classifier = pipeline(
    "text-classification",
    model=model,
    tokenizer=tokenizer
)


In [ ]:
from datetime import datetime

# Generate timestamp
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")  # Format: 20240124_153045
model_path = f"models/{timestamp}/legalbert_model_{timestamp}"

# Save
trainer.save_model(model_path)
tokenizer.save_pretrained(model_path)
print(f"Model saved to: {model_path}")